In [2]:
import triton.language as tl
import torch
import os
os.environ["TRITON_INTERPRET"] = "0"
import softmax1
import torch.nn.functional as F

def softmax_backward(dy, y):
    dot = (dy * y).sum(dim=-1, keepdim=True)
    dx = y * (dy - dot)
    return dx

In [3]:
x = [[1,1,1], [1,2,1]]

def loss_fn(x):
    return torch.sum(x**2)

x_pt = torch.asarray(x, dtype=torch.float32, device="cuda", requires_grad=True)
y_pt = F.softmax(x_pt, dim=-1)
y_pt.retain_grad()
loss_pt = loss_fn(y_pt)
loss_pt.retain_grad()
loss_pt.backward()

x_tr = torch.asarray(x, dtype=torch.float32, device="cuda", requires_grad=True)
y_tr = softmax1.softmax_with_bw(x_tr)
y_tr.retain_grad()
loss_tr = loss_fn(y_tr)
loss_tr.retain_grad()
loss_tr.backward()

assert torch.allclose(x_tr.grad, x_pt.grad)


In [5]:
softmax_backward(y_tr.grad, y_tr)

tensor([[-1.9868e-08, -1.9868e-08, -1.9868e-08],
        [-8.8934e-02,  1.7787e-01, -8.8934e-02]], device='cuda:0',
       grad_fn=<MulBackward0>)

In [6]:
softmax1.softmax_bwd(y_tr.grad, y_tr)

tensor([[-1.9868e-08, -1.9868e-08, -1.9868e-08],
        [-8.8934e-02,  1.7787e-01, -8.8934e-02]], device='cuda:0')

In [5]:
x_tr.grad

tensor([[ 0.2222, -0.1111, -0.1111],
        [ 0.0000,  0.0000,  0.0000]], device='cuda:0')

In [6]:
softmax1.softmax_bwd(y_pt, dy)

NameError: name 'dy' is not defined